In [ ]:
# astrocyte_analysis.py
# Analyzes astrocyte skeletons from pickle files, computing features and saving to CSV.

import os
import numpy as np
import networkx as nx
import pickle
import pandas as pd
from scipy.spatial import ConvexHull
from scipy import linalg
from source.feature_2D import average_bending_energy_curve, graph_to_binary_image_astro, compute_branch_polylines_astro, compute_fractal_dimension_astro

# Define paths and constants
core_path = '../graph_single_mask/'
folders = ['ac_control', 'lat_control', 'vm_control', 'pc_control', 'dm_control']
n = 5
label_map = {
    'ac_control': 0,  # AC
    'dm_control': 1,  # DM
    'lat_control': 2, # LAT
    'vm_control': 3,  # VM
    'pc_control': 4   # PC
}

# Initialize dictionaries to store data
id_data = {folder: [] for folder in folders}
number_of_branches_data = {folder: [] for folder in folders}
mean_branch_length_data = {folder: [] for folder in folders}
edge_density_data = {folder: [] for folder in folders}
mass_data = {folder: {i+1: [] for i in range(4)} for folder in folders}
circular_std_data = {folder: [] for folder in folders}
h_weighted_data = {folder: [] for folder in folders}
phi_data = {folder: [] for folder in folders}
circuity_data = {folder: [] for folder in folders}
betweenness_mean_data = {folder: [] for folder in folders}
diameter_data = {folder: [] for folder in folders}
apl_data = {folder: [] for folder in folders}
assortativity_data = {folder: [] for folder in folders}
algebraic_connectivity_data = {folder: [] for folder in folders}
spectral_entropy_data = {folder: [] for folder in folders}
fractal_dimension_data = {folder: [] for folder in folders}
avg_bending_energy_data = {folder: [] for folder in folders}

# Process each folder and file
for folder in folders:
    folder_path = os.path.join(core_path, folder)
    if not os.path.exists(folder_path):
        print(f"Folder {folder_path} does not exist.")
        continue
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".pkl"):
            id_data[folder].append(file_name)
            file_path = os.path.join(folder_path, file_name)
            with open(file_path, "rb") as file:
                graph = pickle.load(file)

            # Initialize variables
            local_vectors = []
            local_lengths = []
            straight_line_lengths = []
            graph_branch_lengths = []
            all_points = []

            # Compute number of branches
            num_branches = len(graph.edges())
            number_of_branches_data[folder].append(num_branches)

            # Process edges
            for (s, e) in graph.edges():
                pts = graph[s][e]['pts']
                num_px = pts.shape[0]
                branch_length = np.sum(np.sqrt(np.sum(np.diff(pts, axis=0) ** 2, axis=1)))
                graph_branch_lengths.append(branch_length)
                all_points.extend(pts)

                if num_px > n + 1:
                    red_pts = [(pts[0, 1], pts[0, 0])]
                    for i in range((num_px - 1) // n):
                        idx = n * (i + 1)
                        if idx < num_px:
                            red_pts.append((pts[idx, 1], pts[idx, 0]))
                    if (pts[-1, 1], pts[-1, 0]) != red_pts[-1]:
                        red_pts.append((pts[-1, 1], pts[-1, 0]))
                    red_pts = np.array(red_pts)
                    for i in range(len(red_pts) - 1):
                        x0, y0 = red_pts[i]
                        x1, y1 = red_pts[i + 1]
                        dx, dy = x1 - x0, y1 - y0
                        length = np.sqrt(dx**2 + dy**2)
                        if length > 0:
                            local_vectors.append((dx / length, dy / length))
                            local_lengths.append(length)
                    x0, y0 = red_pts[0]
                    x1, y1 = red_pts[-1]
                    straight_length = np.sqrt((x1 - x0)**2 + (y1 - y0)**2)
                    straight_line_lengths.append(straight_length)

            # Compute mean branch length
            mean_branch = np.mean(graph_branch_lengths) if graph_branch_lengths else 0
            mean_branch_length_data[folder].append(mean_branch)

            # Calculate edge density
            all_points = np.array(all_points)
            total_branch_length = sum(graph_branch_lengths)
            unique_points = np.unique(all_points, axis=0)
            if len(unique_points) >= 3:
                hull = ConvexHull(unique_points)
                area = hull.volume
            else:
                area = 0
                print(f"Warning: Not enough unique points for convex hull in {file_name}. Setting area to 0.")
            edge_density = total_branch_length / area if area > 0 else 0
            edge_density_data[folder].append(edge_density)

            # Compute angular metrics and other features
            if len(local_vectors) == 0:
                mass_data[folder][1].append(0)
                mass_data[folder][2].append(0)
                mass_data[folder][3].append(0)
                mass_data[folder][4].append(0)
                circular_std_data[folder].append(0)
                h_weighted_data[folder].append(0)
                phi_data[folder].append(0)
                circuity_data[folder].append(1)
            else:
                angles = np.arctan2([dy for dx, dy in local_vectors], [dx for dx, dy in local_vectors]) % np.pi
                angles_full = 2 * angles
                edge_weights = np.array(local_lengths) / np.sum(local_lengths)
                C_p = np.sum(edge_weights * np.cos(angles_full))
                S_p = np.sum(edge_weights * np.sin(angles_full))
                theta_mean_full = np.arctan2(S_p, C_p) % (2 * np.pi)
                theta_mean = theta_mean_full / 2
                R_p_mean = np.sqrt(C_p**2 + S_p**2)
                circular_std = np.sqrt(-2 * np.log(R_p_mean)) if R_p_mean > 0 else 0
                circular_std_data[folder].append(circular_std)
                rotated_angles = (angles - theta_mean + np.pi/2) % np.pi
                mask_4 = ((rotated_angles >= 0) & (rotated_angles <= np.pi / 8)) | ((rotated_angles >= 7 * np.pi / 8) & (rotated_angles <= np.pi))
                mask_3 = ((rotated_angles > np.pi / 8) & (rotated_angles <= np.pi / 4)) | ((rotated_angles >= 3 * np.pi / 4) & (rotated_angles < 7 * np.pi / 8))
                mask_2 = ((rotated_angles > np.pi / 4) & (rotated_angles <= 3 * np.pi / 8)) | ((rotated_angles >= 5 * np.pi / 8) & (rotated_angles < 3 * np.pi / 4))
                mask_1 = (rotated_angles > 3 * np.pi / 8) & (rotated_angles < 5 * np.pi / 8)
                mass_1 = np.sum(edge_weights * mask_1)
                mass_2 = np.sum(edge_weights * mask_2)
                mass_3 = np.sum(edge_weights * mask_3)
                mass_4 = np.sum(edge_weights * mask_4)
                mass_data[folder][1].append(mass_1)
                mass_data[folder][2].append(mass_2)
                mass_data[folder][3].append(mass_3)
                mass_data[folder][4].append(mass_4)
                n_bins = 18
                bin_edges = np.linspace(0, np.pi, n_bins + 1)
                hist_weighted, _ = np.histogram(rotated_angles, bins=bin_edges, weights=edge_weights, density=False)
                total_weight = np.sum(hist_weighted)
                p_weighted = hist_weighted / total_weight if total_weight > 0 else np.zeros_like(hist_weighted)
                H_weighted = -np.sum([p * np.log(p) if p > 0 else 0 for p in p_weighted])
                H_max = np.log(18)
                phi = 1 - (H_weighted / H_max)**2 if H_max != 0 else 0
                L_net = sum(local_lengths)
                L_g = sum(straight_line_lengths)
                circuity = L_net / L_g if L_g > 0 else 1
                h_weighted_data[folder].append(H_weighted)
                phi_data[folder].append(phi)
                circuity_data[folder].append(circuity)

            # Compute graph metrics
            try:
                bc = nx.betweenness_centrality(graph, weight='weight', normalized=True)
                mean_bc = np.mean(list(bc.values()))
                betweenness_mean_data[folder].append(mean_bc)
            except Exception as e:
                print(f"Betweenness centrality error in {file_name}: {e}")
                betweenness_mean_data[folder].append(0)

            try:
                diameter = nx.diameter(graph, weight='weight')
                diameter_data[folder].append(diameter)
            except Exception as e:
                print(f"Diameter error in {file_name}: {e}")
                diameter_data[folder].append(0)

            try:
                apl = nx.average_shortest_path_length(graph, weight='weight')
                apl_data[folder].append(apl)
            except Exception as e:
                print(f"Average shortest path error in {file_name}: {e}")
                apl_data[folder].append(0)

            try:
                if graph.number_of_edges() < 2:
                    assortativity = 0
                else:
                    assortativity = nx.degree_assortativity_coefficient(graph, weight='weight')
                    assortativity = 0 if np.isnan(assortativity) else assortativity
                assortativity_data[folder].append(assortativity)
            except Exception as e:
                print(f"Assortativity error in {file_name}: {e}")
                assortativity_data[folder].append(0)

            try:
                alg_conn = nx.algebraic_connectivity(graph, weight='weight', method='tracemin_pcg')
                algebraic_connectivity_data[folder].append(alg_conn)
            except Exception as e:
                print(f"Algebraic connectivity error in {file_name}: {e}")
                algebraic_connectivity_data[folder].append(0)

            try:
                L = nx.laplacian_matrix(graph, weight='weight').astype(float).toarray()
                eigenvalues = linalg.eigvalsh(L)
                eigenvalues = np.maximum(eigenvalues, 1e-12)
                probs = eigenvalues / np.sum(eigenvalues)
                spectral_entropy = -np.sum(probs * np.log(probs))
                spectral_entropy_data[folder].append(spectral_entropy)
            except Exception as e:
                print(f"Spectral entropy error in {file_name}: {e}")
                spectral_entropy_data[folder].append(0)

            # Compute fractal dimension
            fractal_dimension_data[folder].append(compute_fractal_dimension_astro(graph))

            # Compute average bending energy
            branch_polylines = compute_branch_polylines_astro(graph)
            branch_avg_bendings = []
            for c in branch_polylines:
                if c.shape[0] >= 3:
                    try:
                        avg_bend = average_bending_energy_curve(c, closed=False)
                        branch_avg_bendings.append(avg_bend)
                    except ValueError as e:
                        print(f"Skipping branch in {file_name}: {e}")
            avg_bending_energy = sum(branch_avg_bendings) if branch_avg_bendings else 0.0
            avg_bending_energy_data[folder].append(avg_bending_energy)

# Compile features into DataFrame and save to CSV
data = []
for folder in folders:
    num_files = len(number_of_branches_data[folder])
    for i in range(num_files):
        row = {
            'id': id_data[folder][i],
            'label': label_map[folder],
            'number_of_branches': number_of_branches_data[folder][i],
            'mean_branch_length': mean_branch_length_data[folder][i],
            'edge_density': edge_density_data[folder][i],
            'mass1': mass_data[folder][1][i],
            'mass2': mass_data[folder][2][i],
            'mass3': mass_data[folder][3][i],
            'mass4': mass_data[folder][4][i],
            'circular_std': circular_std_data[folder][i],
            'weighted_entropy': h_weighted_data[folder][i],
            'phi': phi_data[folder][i],
            'circuity': circuity_data[folder][i],
            'mean_betweenness': betweenness_mean_data[folder][i],
            'diameter': diameter_data[folder][i],
            'apl': apl_data[folder][i],
            'assortativity': assortativity_data[folder][i],
            'algebraic_connectivity': algebraic_connectivity_data[folder][i],
            'spectral_entropy': spectral_entropy_data[folder][i],
            'fractal_dimension': fractal_dimension_data[folder][i],
            'avg_bending_energy': avg_bending_energy_data[folder][i]
        }
        data.append(row)

df = pd.DataFrame(data)
df.to_csv('astro_features.csv', index=False)
print("Feature data saved to 'astro_features.csv'")